In [ ]:
from varphot import nbui as ui

ui.show(ui.header("01", "Vorbereitung — Kalibrierung, Plate-Solve, Vergleichssterne",
    "kalibrierte Bilder · gelöstes Referenzbild (WCS) · Comp-/Check-Sterne (`field_config.json`) in der Sitzung gespeichert",
    flow=[("Schritt 1 — Formular starten", "Die Zellen von oben nach unten ausführen (▶ oder Shift+Enter), bis das **Formular mit den Eingabefeldern** erscheint"),
          ("Schritt 2 — Prüfen", "Meist Defaults übernehmen; der **Hinweiskasten im Formular** zeigt: **grün** = ok · **orange** = prüfen · **rot** = Fehler"),
          ("Schritt 3 — Automatisch", "Weiter mit dem Abschnitt **„Ab hier automatisch“**: bei der nächsten Code-Zelle den Pfeil (▶) öffnen → **„Diese Zelle und alle folgenden“** ausführen")],
    method=ui.current_method(), texts={"mono": "weiter mit NB02", "monorgb": "NB02 je Filter, dann NB08", "osc": "CFA trennen (`tools/split_cfa.py`), NB02 je Kanal, dann NB08", "dslr": "CFA trennen (`tools/split_cfa.py`), NB02 je Kanal, dann NB08"}))

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("01", None, "Felder im Überblick", blocks=[ui.fields([
    ("Kalibrierung", "vorgewählt: *schon kalibriert*", "*Siril-Skript erzeugen und ausführen*"),
    ("Ordner / Referenzbild", "leer lassen", "leer = Defaults aus NB00"),
    ("ASTAP-Radius [°]", "10 lassen", "bei Weitfeld > 10° anpassen"),
    ("Sequenz bis V =", "11 lassen", "bei schwachen Zielen erhöhen"),
    ("Gaia-Optionen", "Häkchen lassen", "Häkchen lassen (brauchen Netz)")], ("Testlauf (LCO)", "Eigene Aufnahmen"))]))

In [ ]:
from varphot import nbui as ui
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

from varphot import Session, load_plan
from varphot import workflow as W
from varphot.hints import bind, nb01_hints
from varphot.siril import find_siril

SESS = Session.current()
PLAN = load_plan(SESS.plan) if SESS.plan and Path(SESS.plan).exists() else None
ui.say(f"Sitzung {SESS.workspace}\n  Ziel {SESS.target} · Methode {SESS.method} · Daten: {SESS.source}" + ("" if PLAN else "\n  ⚠ kein Plan — NB00 ausführen (Aperturen, Filter, Comp-Regeln fehlen sonst)"))

_st, _lay = {"description_width": "150px"}, widgets.Layout(width="620px")
w_cal = widgets.Dropdown(description="Kalibrierung:", options=[("Siril-Skript erzeugen und ausführen", "run"), ("Siril-Skript nur erzeugen (selbst ausführen)", "script"), ("Bilder sind schon kalibriert", "done")],
                         value=os.environ.get("VARPHOT_CAL", "done" if SESS.source == "lco-demo" else "run"), style=_st, layout=_lay)
w_frames = widgets.Text(description="Ordner der Bilder:", value=os.environ.get("VARPHOT_FRAMES_DIR") or SESS.frames_dir, placeholder="leer = process/ der Sitzung", style=_st, layout=_lay)
w_ref = widgets.Text(description="Referenzbild:", value=os.environ.get("VARPHOT_REF", ""), placeholder="leer = scharfes Bild aus NB00, sonst das beste der Reihe", style=_st, layout=_lay)
w_radius = widgets.BoundedFloatText(description="ASTAP-Radius [°]:", value=10.0, min=1.0, max=30.0, style=_st, layout=_lay)
w_maglimit = widgets.FloatSlider(description="Sequenz bis V =", value=float(os.environ.get("VARPHOT_MAGLIMIT", 11.0)), min=8.0, max=14.0, step=0.5, style=_st, layout=_lay)
w_gaia_fill = widgets.Checkbox(description="Comps mit Gaia-Feldsternen auffüllen, wenn die AAVSO-Sequenz zu klein ist", value=True, indent=False, layout=_lay)
w_gaia_check = widgets.Checkbox(description="Gaia-Gegenprüfung (WCS und Sequenzsterne; braucht Netz)", value=os.environ.get("VARPHOT_GAIA_CHECK", "1") != "0", indent=False, layout=_lay)
w_rules = widgets.Text(description="Regeln (JSON):", value=os.environ.get("VARPHOT_RULES", "{}"), placeholder='z. B. {"comp_bv_tol": 0.7}', style=_st, layout=_lay)
w_hint = widgets.HTML()
def _ref_ok():
    r = w_ref.value.strip()
    return bool(r) and (Path(r).exists() or (Path(SESS.workspace) / r).exists())
bind(w_hint, [w_cal, w_ref, w_radius, w_gaia_fill, w_gaia_check],
     lambda: nb01_hints(w_cal.value, find_siril() is not None, SESS.method, w_ref.value, _ref_ok(), w_radius.value, w_gaia_fill.value, w_gaia_check.value, lco=SESS.source == "lco-demo"))
display(widgets.VBox([w_hint, w_cal, w_frames, w_ref, w_radius, w_maglimit, w_gaia_fill, w_gaia_check, w_rules]))

## Ab hier automatisch

Bei der **nächsten Code-Zelle** den Pfeil (▶) öffnen → **„Diese Zelle und alle folgenden“** ausführen. Stoppt mit einer klaren Meldung, wenn etwas fehlt (z. B. Siril-Skript noch nicht gelaufen).

In [ ]:
class Stop(Exception):
    """Sachlicher Abbruch mit Anleitung (kein Programmfehler)."""

if w_cal.value in ("run", "script"):
    CAL = W.calibration_step(SESS, run=(w_cal.value == "run"))
    if CAL["status"] == "keine_lights":
        raise Stop("lights/ der Sitzung ist leer: Light-Bilder (und Darks/Flats/Biases) in die Unterordner legen und erneut starten — oder oben „Bilder sind schon kalibriert“ wählen.")
    ui.say(f"Aufnahmearten: {CAL['kinds']}")
    if CAL["status"] == "skript":
        raise Stop(f"Siril-Skript liegt bereit: {CAL['script']}\n  Ausführen: siril-cli -d \"{SESS.workspace}\" -s \"{CAL['script']}\"  (oder in der Siril-Oberfläche: Skripte → Skript ausführen, Arbeitsordner = Sitzungsordner).\n"
                   "  Danach diese Zelle und die folgenden erneut ausführen." + (f"\n  ({CAL['log']})" if CAL["log"] else ""))
    if CAL["status"] == "fehler":
        raise Stop("Siril lief, aber es entstanden keine kalibrierten Bilder (process/pp_light_*). Protokoll:\n" + CAL["log"][-1500:])
    SESS.update(frames_dir=str(SESS.path("process")))
    ui.say(f"Kalibriert: {len(CAL['frames'])} Bilder in process/")
else:
    SESS.update(frames_dir=w_frames.value.strip() or SESS.frames_dir or str(SESS.path("process")))
FRAMES = W.collect_frames(SESS)
if not FRAMES:
    raise Stop(f"Keine Bilder in {SESS.frames_dir}. Ordner prüfen (FITS, bei Siril: process/pp_light_*).")
ui.say(f"{len(FRAMES)} Bilder der Reihe in {SESS.frames_dir}")

In [ ]:
if w_ref.value.strip():
    REF = Path(w_ref.value.strip()).expanduser()
elif SESS.ref_frame and Path(SESS.ref_frame).exists():
    REF = Path(SESS.ref_frame)
    ui.say("Referenzbild: das scharfe Bild aus NB00")
else:
    REF, REFTAB = W.pick_reference(FRAMES)
    ui.say("Referenzbild automatisch gewählt (meiste scharfe Sterne):"); ui.show(ui.table(REFTAB.round(2)))
RES, SUMM = W.solve_reference(SESS, REF, radius_deg=w_radius.value, fov_deg=(min(PLAN.field_deg) if PLAN and PLAN.field_deg else None))
IMG_R, HDR_R, WCS_R = W.load_reference(SESS)
ui.say(f"Referenz {REF.name}: WCS aus {RES.source}, Skala {SUMM.get('scale_arcsec', '?')} ″/px" + ("" if RES.source == "ASTAP" else "  ⚠ ASTAP hat nicht gelöst — WCS aus dem FITS-Kopf (ungeprüft)"))
for n in RES.notes:
    ui.say("  ⚠", n)
QUAL = W.wcs_quality(WCS_R, IMG_R, HDR_R) if (w_gaia_check.value and RES.source == "ASTAP") else dict(checked=False, note="übersprungen")
if QUAL["checked"]:
    ui.say(f"Gaia-Gegenprobe: Versatz {QUAL['offset_arcsec']:.2f}″, Streuung {QUAL['rms_arcsec']:.2f}″ ({QUAL['n_match']} Treffer) → {QUAL['verdict']}")
else:
    ui.say("Gaia-Gegenprobe:", QUAL["note"])

In [ ]:
FIELD = W.select_field(SESS, PLAN, maglimit=w_maglimit.value, use_gaia_fill=w_gaia_fill.value, rules=json.loads(w_rules.value or "{}"), gaia_check=w_gaia_check.value)
for w in FIELD["warnings"]:
    ui.say("⚠", w)
res = FIELD["res"]
cols = [c for c in ("role", "label", "V", "V_err", "BV", "sep_deg", "reasons") if c in res.colnames]
tab = res[cols].to_pandas()
tab["role"] = pd.Categorical(tab["role"], ["comp", "check", "reserve", "verworfen"], ordered=True)
ui.show(ui.table(tab[tab["role"] != "verworfen"].sort_values(["role", "sep_deg"]).round(3).reset_index(drop=True)))
ui.say("Rollen:", FIELD["counts"], f"| Sequenz-ID (CHART): {FIELD['record']['vsp'].get('chartid')}")
from varphot import plot_comp_field
plot_comp_field(res, FIELD["target"], field_deg=FIELD["field_deg"], pa_deg=FIELD["pa_deg"]); plt.show()

In [ ]:
rows = [("Bilder der Reihe", True, f"{len(FRAMES)} in {SESS.frames_dir}"),
        ("Referenz und WCS", RES.source == "ASTAP", f"{REF.name}: {RES.source}, {SUMM.get('scale_arcsec', '?')} ″/px"),
        ("WCS gegen Gaia", bool(QUAL.get("checked")) and QUAL.get("verdict") == "OK", QUAL.get("verdict") or QUAL.get("note", "")),
        ("Comps", FIELD["counts"]["comp"] >= 4, f"{FIELD['counts']['comp']} Comps, {FIELD['counts']['check']} Check, Sequenz-ID {FIELD['record']['vsp'].get('chartid') or '—'}"),
        ("Gespeichert", True, str(FIELD["path"]))]
ui.show(ui.result("ok" if all(ok for _, ok, _ in rows) else "warn", "Vorbereitung abgeschlossen" if all(ok for _, ok, _ in rows) else "Vorbereitung abgeschlossen, bitte Punkte prüfen",
                  "weiter mit NB02", [ui.status_rows([("ok" if ok else "warn", a, "", b) for a, ok, b in rows])]))